In [ ]:
from tensor_product_space import TensorProductSpace, UnivariateSplineSpace
from hierarchical_mesh import HierarchicalMesh
import numpy as np
from cartesian_mesh import CartesianMesh

In [ ]:
knots = [[-1, -1, -1, -0.5, 0, 0.5, 1, 1, 1], [-1, -1, -0.5, 0., 0.25, 0.5, 1, 1]]
cm = CartesianMesh(knots, 2)
cm.shape

In [ ]:
uss = UnivariateSplineSpace(2, np.array([-1, -1, -1, -0.5,0,0, 0.5, 1, 1, 1]))
uss.bezier

In [ ]:
uss.cell_to_basis_indices([2])

In [ ]:
tps = TensorProductSpace(2, [uss, uss])

In [ ]:
tps2 = tps.refine(dims=[0,1])

In [ ]:
tps2.construct_basis()
tps2.basis_supports

In [ ]:
tps.construct_basis()

In [ ]:
tps.basis[:7]

In [ ]:
A = tps.spaces[0].supports[[0,3,5]]
B = tps.spaces[0].supports[[1,2,4]]

In [ ]:
hm =HierarchicalMesh([np.array([-1, -0.5,0,0.5, 1]), np.array([-1, -0.5,0,0.5, 1])], 2)
hm.refine_in_rectangle(np.array([[-1, 1.], [-0.5, 0.5]]), 1)
hm.refine_in_rectangle(np.array([[-0.25, 0.25], [-0.25, 0.25]]), 2)

In [ ]:
hm.nlevels

In [ ]:
level=1
print(f'active cells of level {level} = \n {hm.aelem_level[level]}')
print(f'deactivated cells of level {level} = \n {hm.delem_level[level]}')

In [ ]:
hm.plot_cells()

In [ ]:
hm.get_children(0, np.array([3, 4, 1]))

In [ ]:
hm.delem_level[1]

In [ ]:
hm.meshes[0].cells[hm.aelem_level[0]]

In [ ]:
hm.meshes[0].cells

In [ ]:
knots = [
        [0, 0, 0, 1.1, 2.9, 3, 3, 3],
        [-3,-3, -1, 0.2, 2, 2],
        #[0,0,0,1,2,3,3,3]
    ]
dim = len(knots)
p = [2, 1]
tsp = TensorProductSpace(knots, p, dim)
tsp.construct_basis()

In [ ]:
def my_construct_basis(knots, dim, degrees):
    knots = np.array(knots)
    idx_start = np.array([
        list(range(len(knots[j]) - degrees[j] - 1)) for j in range(dim)
    ])

    # >>> idx_stop
    # [[4 5 6 7 8]]
    idx_stop = np.array([
        [j + degrees[i] + 2 for j in idx_start[i]] for i in range(dim)
    ])
    # All combinations of starting and stopping indices across all dimensions
    # >>> idx_start_perm
    #[[0]
    # [1]
    # [2]
    # [3]
    # [4]]
    idx_start_perm = np.stack(np.meshgrid(*idx_start), -1).reshape(-1,dim)
    idx_stop_perm = np.stack(np.meshgrid(*idx_stop), -1).reshape(-1,dim)

    n = len(idx_start_perm)
    supports_per_dim = []
    evals_per_dim = []
    basis_per_dim = []
    for j in range(dim):
        starts = knots[j][idx_start_perm[:, j]]
        ends = knots[j][idx_stop_perm[:, j]-1]
        supports_per_dim.append(np.stack((starts, ends), axis=1))

        is_at_end = idx_stop_perm[:, j] == len(knots[j])
        evals_per_dim.append(is_at_end)

        d = degrees[j]
    
        # Create the window offsets: [0, 1, ..., degree+1]
        offsets = np.arange(d + 2)
        
        # Create a grid of indices: Shape (N, degree+2)
        # We broadcast (N, 1) + (1, degree+2)
        grid_indices = idx_start_perm[:, j, None] + offsets[None, :]
        
        # Fancy index into the knot array for dimension j
        # This retrieves the slices for all N functions simultaneously
        basis_per_dim.append(knots[j][grid_indices])

    basis_supports = np.stack(supports_per_dim, axis = 1)
    basis_end_evals = np.stack(evals_per_dim, axis=1).astype(np.int32)
    basis = np.stack(basis_per_dim, axis=1)

    return np.squeeze(basis_supports), np.squeeze(basis_end_evals), np.squeeze(basis)

In [ ]:
tsp.mesh.cells

In [ ]:
tsp.basis_supports[[3,5, 8]]

In [ ]:
support = tsp.get_cells([3,5, 8])

In [ ]:
support

In [ ]:
support_cells = tsp.basis_to_cell([0,1,7])
print(tsp.mesh.cells[support_cells])